In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import RepeatVector, Concatenate, Dense, Softmax, Dot
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Lambda, TimeDistributed, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Dataset

In [2]:
from faker import Faker
from babel.dates import format_date
import random
from tqdm import tqdm

faker = Faker()
Faker.seed(12345)
random.seed(12345)

In [3]:
FORMATS = {
    'short': 1,
    'medium': 1,
    'long': 1,
    'full': 5,
    'd MMM YYY': 1,
    'dd MMM YYY': 1,
    'd MMMM YYY': 1,
    'd MMM, YYY': 1,
    'd MMMM, YYY': 1,
    'dd.MM.YY': 1,
    'MMMM d YYY': 1,
    'MMMM d, YYY': 1
}

formats = list(FORMATS.keys())
weights = list(FORMATS.values())

date_obj = faker.date_object()
random_format = random.choices(formats, weights=weights, k=1)  # returns list of k elements
formatted_date = format_date(date_obj, random_format[0], locale="en_US")
iso_date = date_obj.isoformat()  # convert into ISO format YYYY-MM-DD

print(date_obj, type(date_obj))
print(random_format)
print(formatted_date)
print(iso_date, type(iso_date))

1998-05-09 <class 'datetime.date'>
['full']
Saturday, May 9, 1998
1998-05-09 <class 'str'>


In [4]:
def generate_date():
    date_object  = faker.date_object()
    human_readable_date  = format_date(date_object , format=random.choices(formats, weights=weights, k=1)[0], locale='en_US')
    human_readable_date  = human_readable_date.lower()
    machine_readable_date  = date_object .isoformat()
    return human_readable_date, machine_readable_date

In [5]:
def load_dataset(m):
    human_vocab = set()
    machine_vocab = set()
    dataset = []

    for i in tqdm(range(m)):
        h, m = generate_date()
        # set.add(), if string, it will add the whole string
        # set.update(), if string, it will iterate for each character and add each to set
        human_vocab.update(h)
        machine_vocab.update(m)
        dataset.append((h, m))

    human_vocab_to_idx = dict(zip(
        ['<pad>', '<unk>'] + sorted(human_vocab), 
        list(range(len(human_vocab)+2))
    ))

    idx_to_machine_vocab = dict(enumerate(sorted(machine_vocab)))
    machine_vocab_to_idx = {vocab:idx for idx, vocab in idx_to_machine_vocab.items()}
    
    return dataset, human_vocab_to_idx, machine_vocab_to_idx, idx_to_machine_vocab

In [6]:
def convert_string_to_indices(string, vocab_to_idx, length):
    if len(string) > length:
        string = string[:length]
        
    indices = []
    for char in string:
        if char in vocab_to_idx:
            indices.append(vocab_to_idx[char])
        else:
            indices.append(vocab_to_idx['<unk>'])

    while len(indices) < length:
        indices.append(vocab_to_idx['<pad>'])

    return indices

In [7]:
def preprocess_dataset(dataset, human_vocab_to_idx, machine_vocab_to_idx, input_length, output_length):
    human_dates, machine_dates = zip(*dataset)  # zip(*list(tuple)) returns list of element 1 in tuple, list of element 2 in tuple

    X = np.array([convert_string_to_indices(dt, human_vocab_to_idx, input_length) for dt in human_dates])       # (m, input_length)
    Y = np.array([convert_string_to_indices(dt, machine_vocab_to_idx, output_length) for dt in machine_dates])  # (m, output_length)

    X_onehot = to_categorical(X, num_classes=len(human_vocab_to_idx))    # (m, input_length, human_vocab_size)
    Y_onehot = to_categorical(Y, num_classes=len(machine_vocab_to_idx))  # (m, output_length, machine_vocab_size)
    
    return X, Y, X_onehot, Y_onehot

In [8]:
n_samples = 10000
dataset, human_vocab_to_idx, machine_vocab_to_idx, idx_to_machine_vocab = load_dataset(n_samples)

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [00:01<00:00, 5788.68it/s]


In [9]:
dataset[:10]

[('11/10/19', '2019-11-10'),
 ('10.09.70', '1970-09-10'),
 ('monday, august 19, 2024', '2024-08-19'),
 ('saturday, april 28, 1990', '1990-04-28'),
 ('thursday, january 26, 1995', '1995-01-26'),
 ('07 mar 1983', '1983-03-07'),
 ('may 22, 1988', '1988-05-22'),
 ('jul 8, 2008', '2008-07-08'),
 ('wednesday, september 8, 1999', '1999-09-08'),
 ('1 jan 1981', '1981-01-01')]

In [10]:
print(human_vocab_to_idx)

{'<pad>': 0, '<unk>': 1, ' ': 2, ',': 3, '.': 4, '/': 5, '0': 6, '1': 7, '2': 8, '3': 9, '4': 10, '5': 11, '6': 12, '7': 13, '8': 14, '9': 15, 'a': 16, 'b': 17, 'c': 18, 'd': 19, 'e': 20, 'f': 21, 'g': 22, 'h': 23, 'i': 24, 'j': 25, 'l': 26, 'm': 27, 'n': 28, 'o': 29, 'p': 30, 'r': 31, 's': 32, 't': 33, 'u': 34, 'v': 35, 'w': 36, 'y': 37}


In [11]:
print(machine_vocab_to_idx)

{'-': 0, '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6, '6': 7, '7': 8, '8': 9, '9': 10}


In [12]:
maxSeqLen_x = max(len(x) for x, _ in dataset) 
maxSeqLen_y = max(len(y) for _, y in dataset)

maxSeqLen_x, maxSeqLen_y

(29, 10)

In [13]:
Tx = 30
Ty = 10
X, Y, Xoh, Yoh = preprocess_dataset(dataset, human_vocab_to_idx, machine_vocab_to_idx, Tx, Ty)

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)
print("Xoh.shape:", Xoh.shape)
print("Yoh.shape:", Yoh.shape)

X.shape: (10000, 30)
Y.shape: (10000, 10)
Xoh.shape: (10000, 30, 38)
Yoh.shape: (10000, 10, 11)


# Model

In [14]:
def get_context_vector(s_prev, a, layers):
    """
    Performs ...
    
    Arguments:
    s_prev -- previous hidden state of the LSTM decoder, numpy-array of shape (m, n_s)
    a -- all hidden states of the Bi-LSTM encoder, numpy-array of shape (m, Tx, 2*n_a)
    
    Returns:
    context_vector -- context vector, input of the LSTM Decoder
    """
    (attn_repeat_layer, attn_concat_layer, attn_dense_L1, attn_dense_L2, attn_weights_layer, attn_scores_layer) = layers

    s_prev_repeated = attn_repeat_layer(s_prev)            # (m, Tx, n_s)
    as_concated = attn_concat_layer([a, s_prev_repeated])  # (m, Tx, 2*n_a + n_s)
    
    dense_1 = attn_dense_L1(as_concated)  # (m, Tx, n_hidden_neurons)
    dense_2 = attn_dense_L2(dense_1)      # (m, Tx, 1)

    attention_weights = attn_weights_layer(dense_2)  # apply softmax for the axis Tx, output_shape (m, Tx, 1)
    
    context_vector = attn_scores_layer([attention_weights, a])  
    # Dot(axis=1) means [1 * (2*n_a)] for each Tx, then sum along the axis Tx, output_shape (m, 1, 2*n_a)

    return context_vector

In [15]:
def build_seq2seq_model(
    input_seq_length, output_seq_length,
    human_vocab_size, machine_vocab_size,
    encoder_hidden_dim, decoder_hidden_dim,
    dropout_rate=0.3
):
    
    # For simplicity, no masking here
    encoder_input = Input(shape=(input_seq_length, human_vocab_size), name="encoder_input")  # (m, Tx, human_vocab_size)

    # Encoder 
    encoder_hidden_states = Bidirectional(
        LSTM(encoder_hidden_dim, return_sequences=True, dropout=dropout_rate, name="encoder_lstm"), name="bidirectional_encoder"
    )(encoder_input)  # (m, Tx, 2*n_a)
    
    # Init hidden state & cell state for decoder
    s_prev = Lambda(lambda x: tf.zeros((tf.shape(x)[0], decoder_hidden_dim)), name="decoder_initial_hidden_state")(encoder_input)
    c_prev = Lambda(lambda x: tf.zeros((tf.shape(x)[0], decoder_hidden_dim)), name="decoder_initial_cell_state")(encoder_input)

    # Decoder Layer
    decoder_lstm = LSTM(decoder_hidden_dim, return_sequences=False, return_state=True, dropout=dropout_rate, name=f"decoder_lstm")
    # if return_state=True, it will return 3 results: output(s), last_hidden_state, last_cell_state
    # if return_sequences=True, the first one will output [s1, s2, s3, ...], but we dont need it
    # if return_sequences=False, the first one will output the last hidden state, which is the same as the second

    # Attention Layer
    attn_repeat_layer = RepeatVector(input_seq_length, name="attn_repeat_layer")
    attn_concat_layer = Concatenate(axis=-1, name="attn_concated_layer")
    attn_dense_L1 = Dense(32, activation="tanh", name="attn_dense_L1")
    attn_dense_L2 = Dense(1, activation="linear", name="attn_dense_L2")  # NO activation="softmax" here, because it only computes for the last axis
    attn_weights_layer = Softmax(axis=1, name="attn_weights_layer")
    attn_scores_layer = Dot(axes=1, name="attn_scores_layer")

    # Output Layer
    output_dropout_layer = Dropout(dropout_rate, name="output_dropout_layer")
    output_dense_layer = Dense(machine_vocab_size, activation="softmax", name="output_dense_layer")  

    
    # Run decoder + attention + output
    all_probs = []
    
    for t in range(output_seq_length):
        context_vector = get_context_vector(
            s_prev=s_prev, 
            a=encoder_hidden_states,
            layers=[attn_repeat_layer, attn_concat_layer, attn_dense_L1, attn_dense_L2, attn_weights_layer, attn_scores_layer]
        )  # (m, 1, 2*n_a)

        # IN FACT, the context vector is not fed into the LSTM Decoder. Instead, it is concatenated with the s_t and passed to a Dense layer.
        # During TRAINING, the actual input to the LSTM Decoder is the ground-truth token. During INFERENCE, it is the previously predicted token.
        decoder_output, s, c = decoder_lstm(context_vector, initial_state=[s_prev, c_prev])

        decoder_output = output_dropout_layer(decoder_output)  # (m, n_s)
        
        probs = output_dense_layer(decoder_output)  # (m, machine_vocab_size)
        all_probs.append(probs)
        
        s_prev = s
        c_prev = c

    all_probs = Lambda(lambda x: tf.stack(x, axis=1), name="stacked_output")(all_probs)  # (m, Ty, machine_vocab_size)

    # Encoder-Decoder
    model = Model(inputs=encoder_input, outputs=all_probs, name="Seq2Seq_Attention_Model")
    return model

In [16]:
Tx, Ty = 30, 10
n_a, n_s = 16, 32
# bi_lstm will double n_a

model = build_seq2seq_model(
    input_seq_length=Tx,      
    output_seq_length=Ty,   
    human_vocab_size=len(human_vocab_to_idx),
    machine_vocab_size=len(machine_vocab_to_idx),
    encoder_hidden_dim=n_a,   
    decoder_hidden_dim=n_s
)

model.summary()

Model: "Seq2Seq_Attention_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)    │ (None, 30, 38)            │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_initial_hidden_state  │ (None, 32)                │               0 │ encoder_input[0][0]        │
│ (Lambda)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional_encoder         │ (None, 30, 32)            │           7,040 │ encoder_input[0][0]        │
│ (Bidirectional)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attn_repeat_layer             │ (None, 30, 32)            │               0 │ decoder_initial_hidden_st… │
│ (RepeatVector)                │                           │                 │ decoder_lstm[0][1],        │
│                               │                           │                 │ decoder_lstm[1][1],        │
│                               │                           │                 │ decoder_lstm[2][1],        │
│                               │                           │                 │ decoder_lstm[3][1],        │
│                               │                           │                 │ decoder_lstm[4][1],        │
│                               │                           │                 │ decoder_lstm[5][1],        │
│                               │                           │                 │ decoder_lstm[6][1],        │
│                               │                           │                 │ decoder_lstm[7][1],        │
│                               │                           │                 │ decoder_lstm[8][1]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attn_concated_layer           │ (None, 30, 64)            │               0 │ bidirectional_encoder[0][… │
│ (Concatenate)                 │                           │                 │ attn_repeat_layer[0][0],   │
│                               │                           │                 │ bidirectional_encoder[0][… │
│                               │                           │                 │ attn_repeat_layer[1][0],   │
│                               │                           │                 │ bidirectional_encoder[0][… │
│                               │                           │                 │ attn_repeat_layer[2][0],   │
│                               │                           │                 │ bidirectional_encoder[0][… │
│                               │                           │                 │ attn_repeat_layer[3][0],   │
│                               │                           │                 │ bidirectional_encoder[0][… │
│                               │                           │                 │ attn_repeat_layer[4][0],   │
│                               │                           │                 │ bidirectional_encoder[0][… │
│                               │                           │                 │ attn_repeat_layer[5][0],   │
│                               │                           │                 │ bidirectional_encoder[0][… │
│                               │                           │                 │ attn_repeat_layer[6][0],   │
│                               │                           │               

 Total params: 17,836 (69.67 KB)

 Trainable params: 17,836 (69.67 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
optimizer = Adam(learning_rate=0.005, beta_1=0.9, beta_2=0.999)

model.compile(
    optimizer=optimizer, 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


early_stop = EarlyStopping(
    monitor='accuracy',
    patience=3,
    restore_best_weights=True,
    min_delta=1e-3
)

In [18]:
Yoh_swapped = Yoh.swapaxes(0, 1)  # (m, Tx, vocab_size) --> (Tx, m, vocab_size)
true_outputs = list(Yoh_swapped)

Yoh_swapped.shape, true_outputs[0].shape

((10, 10000, 11), (10000, 11))

In [19]:
# use this if you not using tf.stack in model, model returns list of outputs
# history = model.fit(Xoh, true_outputs, epochs=5, batch_size=128)

In [20]:
history = model.fit(
    Xoh, Yoh, 
    epochs=50, 
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 59s 67ms/step - accuracy: 0.2710 - loss: 2.0592
Epoch 2/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 78ms/step - accuracy: 0.6064 - loss: 1.0432
Epoch 3/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 68ms/step - accuracy: 0.7191 - loss: 0.7712
Epoch 4/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 23s 81ms/step - accuracy: 0.8105 - loss: 0.5524
Epoch 5/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 18s 112ms/step - accuracy: 0.8630 - loss: 0.4223
Epoch 6/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 96ms/step - accuracy: 0.8907 - loss: 0.3416
Epoch 7/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 66ms/step - accuracy: 0.9066 - loss: 0.2980
Epoch 8/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 78ms/step - accuracy: 0.9202 - loss: 0.2574
Epoch 9/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 67ms/step - accuracy: 0.9327 - loss: 0.2220
Epoch 10/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step - accuracy: 0.9388 - loss: 0.2029
Epoch 11/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 67ms/step - accuracy: 0.9431 - loss: 0.1875
Epoch 12/50
157/157 ━━━━━━━━━

In [21]:
examples = [
    '3 May 1979', 
    '5 April 09', 
    '21th of August 2016', 
    'Tue 10 Jul 2007', 
    'Saturday May 9 2018', 
    'March 3 2001', 
    'March 3rd 2001', 
    '1 March 2001'
]

examples_X = np.array([
    convert_string_to_indices(s.lower(), human_vocab_to_idx, Tx)
    for s in examples
])

examples_Xoh = to_categorical(examples_X, num_classes=len(human_vocab_to_idx))
examples_Xoh.shape

(8, 30, 38)

In [22]:
preds = model.predict(examples_Xoh)
preds.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step


(8, 10, 11)

In [23]:
def convert_oh_to_string(oh, idx_to_vocab):
    indices = np.argmax(oh, axis=-1)  # (Ty, vocab_size) --> (Ty, )
    chars = [idx_to_vocab[i] for i in indices]
    return ''.join(chars).replace('<pad>', '').strip()

In [24]:
preds_str = [convert_oh_to_string(p, idx_to_machine_vocab) for p in preds]

for i, j in zip(examples, preds_str):
    print(f"{i:30s} → {j}")

3 May 1979                     → 1979-05-03
5 April 09                     → 2009-04-05
21th of August 2016            → 2016-08-01
Tue 10 Jul 2007                → 2007-07-10
Saturday May 9 2018            → 2018-05-09
March 3 2001                   → 2001-03-03
March 3rd 2001                 → 2001-03-03
1 March 2001                   → 2001-03-01


# Visualizing Attention